# Notebook 1 — Data Generation & Schema Validation
## Predictive Logistics Engine — Mumbai Fleet Simulation

**What this notebook does:**
- Generates 3 simulated event streams (GPS, Sensor, Package Manifest) for 100 Mumbai delivery vans over 7 days
- Validates every row against the schema definitions
- Produces sample CSVs used by Notebooks 2 and 3
- Shows the Friday variance spike embedded in the raw data

**System context:** In production, these streams flow through Azure IoT Hub → Schema Registry (Avro validation) → Azure Event Hubs at 3,000+ events/sec across 5,000 vans.


In [ ]:
# Install dependencies
!pip install pandas numpy matplotlib seaborn folium h3 --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import uuid, random, math, json, os
from datetime import datetime, timedelta

np.random.seed(42)
random.seed(42)

# Plot style
plt.rcParams.update({
    'figure.facecolor': '#0A1628',
    'axes.facecolor':   '#132040',
    'axes.edgecolor':   '#1E3A5F',
    'axes.labelcolor':  '#94A3B8',
    'xtick.color':      '#94A3B8',
    'ytick.color':      '#94A3B8',
    'text.color':       '#FFFFFF',
    'grid.color':       '#1E3A5F',
    'grid.alpha':       0.5,
    'lines.linewidth':  2,
})

print("Libraries loaded.")

## 1. Mumbai Geography Configuration

Real coordinates for key Mumbai delivery zones, medical destinations (LIFE_CRITICAL drop points), and van depots.


In [ ]:
ZONES = {
    "BKC":         {"lat": 19.0596, "lon": 72.8656, "label": "Bandra Kurla Complex"},
    "Dharavi":     {"lat": 19.0418, "lon": 72.8544, "label": "Dharavi"},
    "Andheri":     {"lat": 19.1197, "lon": 72.8464, "label": "Andheri West"},
    "Lower_Parel": {"lat": 18.9938, "lon": 72.8258, "label": "Lower Parel"},
    "Worli":       {"lat": 19.0178, "lon": 72.8178, "label": "Worli"},
    "Dadar":       {"lat": 19.0180, "lon": 72.8417, "label": "Dadar"},
    "Powai":       {"lat": 19.1176, "lon": 72.9060, "label": "Powai"},
    "Bandra":      {"lat": 19.0544, "lon": 72.8402, "label": "Bandra West"},
    "Kurla":       {"lat": 19.0728, "lon": 72.8826, "label": "Kurla"},
    "Ghatkopar":   {"lat": 19.0862, "lon": 72.9077, "label": "Ghatkopar"},
    "Chembur":     {"lat": 19.0620, "lon": 72.9001, "label": "Chembur"},
    "Malad":       {"lat": 19.1865, "lon": 72.8486, "label": "Malad West"},
}

MEDICAL_DESTINATIONS = [
    {"name": "Hinduja Hospital, Mahim",      "lat": 19.0414, "lon": 72.8370},
    {"name": "Kokilaben Hospital, Andheri",  "lat": 19.1362, "lon": 72.8264},
    {"name": "Lilavati Hospital, Bandra",    "lat": 19.0524, "lon": 72.8259},
    {"name": "KEM Hospital, Parel",          "lat": 18.9960, "lon": 72.8409},
    {"name": "Nanavati Hospital, Vile Parle","lat": 19.0990, "lon": 72.8388},
]

DEPOTS = [
    {"name": "Andheri Depot",  "lat": 19.1197, "lon": 72.8464},
    {"name": "Dadar Depot",    "lat": 19.0180, "lon": 72.8417},
    {"name": "Kurla Depot",    "lat": 19.0728, "lon": 72.8826},
]

N_VANS, N_DAYS, INTERVAL_SEC = 100, 7, 300
START_DATE = datetime(2024, 3, 11)  # Monday
print(f"Config: {N_VANS} vans, {N_DAYS} days, {INTERVAL_SEC}s interval")
print(f"Total GPS events expected: {N_VANS * N_DAYS * (14*3600//INTERVAL_SEC):,}")

## 2. Simulation Functions

Key design decisions embedded in the simulation:
- **10% of vans** have faulty GPS (jitter vans) — random heading, gps_accuracy_m > 40m
- **Van VAN-0042** has a PSI drop event on Day 3 at 14:30 (the Van #402 interview scenario)
- **Friday 14:00–19:00** injects 4× std-dev multiplier on travel times


In [ ]:
def simple_geohash(lat, lon, precision=6):
    lat_q = round(lat * precision, 0) / precision
    lon_q = round(lon * precision, 0) / precision
    return f"{lat_q:.4f}_{lon_q:.4f}"

def get_speed_kmh(dt, is_friday_afternoon):
    hour = dt.hour
    if 7 <= hour <= 10 or 17 <= hour <= 21:
        base = np.random.normal(15, 4)
    elif hour >= 22 or hour <= 5:
        base = np.random.normal(45, 5)
    else:
        base = np.random.normal(28, 6)
    if is_friday_afternoon:
        base = np.random.normal(base * 0.7, 4 * 4.0)  # 400% variance
    return float(np.clip(base, 3.0, 70.0))

def create_fleet():
    vans = []
    for i in range(N_VANS):
        depot = DEPOTS[i % len(DEPOTS)]
        vans.append({
            "van_id":        f"VAN-{str(i+1).zfill(4)}",
            "depot":         depot,
            "lat":           depot["lat"] + np.random.normal(0, 0.005),
            "lon":           depot["lon"] + np.random.normal(0, 0.005),
            "heading":       random.uniform(0, 360),
            "is_jitter":     (i % 10 == 7),
            "fuel_pct":      random.uniform(60, 100),
            "tyre_psi":      [random.uniform(33, 36) for _ in range(4)],
            "engine_temp":   random.uniform(85, 95),
            "odometer":      random.randint(10000, 80000),
            "firmware":      random.choice(["v2.3.1","v2.4.1","v2.4.2"]),
            "is_van402":     (i == 41),  # VAN-0042
        })
    return vans

print("Simulation functions ready.")

In [ ]:
def run_simulation():
    vans = create_fleet()
    gps_rows, sensor_rows, manifest_rows = [], [], []

    for day in range(N_DAYS):
        day_dt   = START_DATE + timedelta(days=day)
        day_name = day_dt.strftime("%A")
        is_fri   = day_dt.weekday() == 4
        print(f"  Day {day+1}/7: {day_name} {day_dt.strftime('%d %b')} {'← FRIDAY VARIANCE SPIKE' if is_fri else ''}")

        # Package manifests at day start
        for van in vans:
            for _ in range(8):
                r = random.random()
                if r < 0.05:
                    prio = "LIFE_CRITICAL"; dest = random.choice(MEDICAL_DESTINATIONS); sla_h = random.uniform(0.5, 2)
                elif r < 0.25:
                    prio = "PREMIUM"; z = random.choice(list(ZONES.values())); dest = {"name":z["label"],"lat":z["lat"],"lon":z["lon"]}; sla_h = random.uniform(2,4)
                else:
                    prio = "STANDARD"; z = random.choice(list(ZONES.values())); dest = {"name":z["label"],"lat":z["lat"],"lon":z["lon"]}; sla_h = random.uniform(4,8)
                load_t = day_dt + timedelta(hours=random.uniform(7,10))
                sla_t  = load_t + timedelta(hours=sla_h)
                manifest_rows.append({
                    "event_id":str(uuid.uuid4()),"van_id":van["van_id"],
                    "timestamp_utc":int(load_t.timestamp()*1000),
                    "event_type":"LOADED","package_id":str(uuid.uuid4()),
                    "priority_class":prio,
                    "delivery_sla_utc":int(sla_t.timestamp()*1000),
                    "dest_lat":round(dest["lat"],6),"dest_lon":round(dest["lon"],6),
                    "dest_name":dest["name"],"time_to_sla_sec":int(sla_h*3600),
                    "weight_kg":round(random.uniform(0.5,25),2),
                    "requires_cold_chain": prio=="LIFE_CRITICAL" and random.random()<0.4,
                    "schema_version":1,
                })

        t   = day_dt.replace(hour=7,  minute=0, second=0)
        end = day_dt.replace(hour=21, minute=0, second=0)

        while t < end:
            h   = t.hour
            fri_pm = is_fri and 14 <= h <= 19

            for van in vans:
                van402_event = van["is_van402"] and day==2 and h==14 and t.minute==30
                speed = get_speed_kmh(t, fri_pm)

                # Update position
                tz = random.choice(list(ZONES.values()))
                dlat = tz["lat"] - van["lat"]; dlon = tz["lon"] - van["lon"]
                d = math.sqrt(dlat**2+dlon**2)
                if d > 0.001:
                    dk = (speed*(INTERVAL_SEC/3600))/111.0
                    van["lat"] += (dlat/d)*dk + np.random.normal(0,0.0001)
                    van["lon"] += (dlon/d)*dk*(1/math.cos(math.radians(van["lat"]))) + np.random.normal(0,0.0001)
                    van["heading"] = (math.degrees(math.atan2(dlon,dlat))+360)%360

                raw_lat, raw_lon = van["lat"], van["lon"]
                accuracy = 5.0
                if van["is_jitter"]:
                    raw_lat += np.random.normal(0, 0.00045)
                    raw_lon += np.random.normal(0, 0.00045)
                    accuracy = random.uniform(45,80)
                    van["heading"] = random.uniform(0,360)

                # Update sensors
                van["tyre_psi"] = [p - random.uniform(0,0.005) for p in van["tyre_psi"]]
                van["engine_temp"] = float(np.clip(van["engine_temp"] + np.random.normal(0,0.5), 80, 105))
                van["fuel_pct"]    = max(5.0, van["fuel_pct"] - random.uniform(0.01,0.05))
                van["odometer"]   += random.randint(0,1)
                if van402_event: van["tyre_psi"][0] -= 5.0

                psi_min = min(van["tyre_psi"])
                flags   = (1 if van["engine_temp"]>100 else 0)|(2 if psi_min<28 else 0)|(4 if van["fuel_pct"]<15 else 0)
                ts_ms   = int(t.timestamp()*1000)
                ing_ms  = ts_ms + random.randint(200,800)

                health = min(100, 0.4*min(100,max(0,(psi_min-20)/16*100))
                            +0.4*min(100,max(0,(110-van["engine_temp"])/30*100))
                            +0.2*van["fuel_pct"])

                gps_rows.append({
                    "event_id":str(uuid.uuid4()),"van_id":van["van_id"],
                    "timestamp_utc":ts_ms,"ingested_utc":ing_ms,
                    "lat":round(raw_lat,6),"lon":round(raw_lon,6),
                    "geo_hash":simple_geohash(raw_lat,raw_lon),
                    "speed_kmh":round(speed,2),"heading_deg":round(van["heading"],1),
                    "gps_accuracy_m":round(accuracy,1),
                    "satellites_locked":random.randint(6,12) if not van["is_jitter"] else random.randint(3,6),
                    "is_jitter_van":van["is_jitter"],"day_name":day_name,
                    "is_friday_afternoon":fri_pm,"schema_version":1,
                })
                sensor_rows.append({
                    "event_id":str(uuid.uuid4()),"van_id":van["van_id"],
                    "timestamp_utc":ts_ms,"ingested_utc":ing_ms,
                    "tyre_psi_fl":round(van["tyre_psi"][0],2),
                    "tyre_psi_fr":round(van["tyre_psi"][1],2),
                    "tyre_psi_rl":round(van["tyre_psi"][2],2),
                    "tyre_psi_rr":round(van["tyre_psi"][3],2),
                    "tyre_psi_min":round(psi_min,2),
                    "engine_temp_c":round(van["engine_temp"],1),
                    "fuel_level_pct":round(van["fuel_pct"],1),
                    "speed_kmh":round(speed,2),"odometer_km":van["odometer"],
                    "alert_flags":flags,"health_score":round(health,1),
                    "safe_to_continue": psi_min>=26 and van["engine_temp"]<105 and health>30,
                    "is_van402":van["is_van402"],"firmware_version":van["firmware"],
                    "day_name":day_name,"schema_version":1,
                })
            t += timedelta(seconds=INTERVAL_SEC)

    return pd.DataFrame(gps_rows), pd.DataFrame(sensor_rows), pd.DataFrame(manifest_rows)

print("Running simulation...")
gps_df, sensor_df, manifest_df = run_simulation()
print(f"\nGPS rows:      {len(gps_df):,}")
print(f"Sensor rows:   {len(sensor_df):,}")
print(f"Manifest rows: {len(manifest_df):,}")

## 3. Schema Validation

In [ ]:
def validate_schema(df, name, required_cols, type_checks):
    print(f"\n{'='*50}")
    print(f"Schema validation: {name}")
    print(f"{'='*50}")
    missing = [c for c in required_cols if c not in df.columns]
    if missing:
        print(f"  FAIL — missing columns: {missing}")
    else:
        print(f"  PASS — all {len(required_cols)} required columns present")
    nulls = df[required_cols].isnull().sum()
    null_cols = nulls[nulls > 0]
    if len(null_cols) == 0:
        print(f"  PASS — zero null values in required fields")
    else:
        print(f"  WARN — nulls found: {null_cols.to_dict()}")
    print(f"  Rows: {len(df):,}  |  Columns: {len(df.columns)}")
    for col, check, msg in type_checks:
        result = check(df[col])
        status = "PASS" if result else "FAIL"
        print(f"  {status} — {col}: {msg}")

validate_schema(gps_df, "GPS Telemetry",
    ["event_id","van_id","timestamp_utc","lat","lon","geo_hash","speed_kmh","heading_deg","gps_accuracy_m"],
    [
        ("lat",  lambda s: s.between(18.8, 19.4).all(),  "all lats within Mumbai bounds"),
        ("lon",  lambda s: s.between(72.7, 73.1).all(),  "all lons within Mumbai bounds"),
        ("speed_kmh", lambda s: (s>=0).all(),             "no negative speeds"),
        ("heading_deg", lambda s: s.between(0,360).all(), "heading 0-360"),
    ])

validate_schema(sensor_df, "Sensor Telemetry",
    ["event_id","van_id","timestamp_utc","tyre_psi_min","engine_temp_c","alert_flags","health_score","safe_to_continue"],
    [
        ("tyre_psi_min",   lambda s: (s>0).all(),              "positive PSI values"),
        ("health_score",   lambda s: s.between(0,100).all(),   "health score 0-100"),
        ("engine_temp_c",  lambda s: s.between(60,120).all(),  "realistic temperature range"),
        ("fuel_level_pct", lambda s: s.between(0,100).all(),   "fuel 0-100%"),
    ])

validate_schema(manifest_df, "Package Manifest",
    ["event_id","van_id","package_id","priority_class","delivery_sla_utc","dest_lat","dest_lon","time_to_sla_sec"],
    [
        ("priority_class", lambda s: s.isin(["LIFE_CRITICAL","PREMIUM","STANDARD"]).all(), "valid priority classes"),
        ("time_to_sla_sec", lambda s: (s>0).all(), "positive SLA seconds"),
        ("weight_kg", lambda s: (s>0).all(), "positive weights"),
    ])

## 4. Priority Distribution Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Priority distribution
colors = {'LIFE_CRITICAL':'#EF4444', 'PREMIUM':'#F59E0B', 'STANDARD':'#10B981'}
priority_counts = manifest_df['priority_class'].value_counts()
bars = axes[0].bar(priority_counts.index,
                   priority_counts.values,
                   color=[colors[p] for p in priority_counts.index],
                   width=0.5, edgecolor='none')
for bar, val in zip(bars, priority_counts.values):
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+20,
                 f'{val:,}\n({val/len(manifest_df)*100:.1f}%)',
                 ha='center', va='bottom', fontsize=10, color='white')
axes[0].set_title('Package priority distribution — 7 days', fontsize=13, pad=12)
axes[0].set_ylabel('Package count')
axes[0].set_ylim(0, priority_counts.max() * 1.2)
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_axisbelow(True)

# GPS jitter vans vs normal
jitter_counts = gps_df.drop_duplicates('van_id').groupby('is_jitter_van').size()
labels = ['Normal GPS (90%)', 'Jitter GPS (10%)']
wedge_colors = ['#0EA5E9','#EF4444']
axes[1].pie(jitter_counts.values, labels=labels, colors=wedge_colors,
            autopct='%1.0f%%', startangle=90,
            textprops={'color':'white','fontsize':11},
            wedgeprops={'linewidth':2, 'edgecolor':'#0A1628'})
axes[1].set_title('Van GPS quality split', fontsize=13, pad=12)

plt.tight_layout(pad=2)
plt.savefig('priority_distribution.png', dpi=150, bbox_inches='tight', facecolor='#0A1628')
plt.show()
print("Chart 1 saved.")

## 5. Save Data Files

In [ ]:
os.makedirs('data/sample', exist_ok=True)
gps_df.to_csv('data/sample/gps_telemetry.csv',      index=False)
sensor_df.to_csv('data/sample/sensor_telemetry.csv', index=False)
manifest_df.to_csv('data/sample/package_manifest.csv', index=False)

print("Files saved:")
print(f"  data/sample/gps_telemetry.csv      — {len(gps_df):,} rows")
print(f"  data/sample/sensor_telemetry.csv   — {len(sensor_df):,} rows")
print(f"  data/sample/package_manifest.csv   — {len(manifest_df):,} rows")
print("\nReady for Notebook 2 — Intelligence Algorithms")